<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/09_sandboxes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 · Sandboxes: a real computer for the agent

Every backend so far has been a *filing cabinet* — somewhere to put text. A **sandbox** is a
filing cabinet with a CPU: a real Linux box the agent can install packages on, run shell
commands in, and serve HTTP from.

By the end you will ask an agent for a report, and it will hand you back a **URL you can click**
that renders the page it wrote.

**New in this lesson:** `LangSmithSandbox`, the `execute` tool, `sb.service()`, and download links

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.
>
> Running locally instead? `uv sync`, then put the key in a `.env` at the repo root.

In [ ]:
# --- snippet:setup v2 ---
import contextlib
import os

try:
    from google.colab import userdata

    %pip install -qq --progress-bar off \
      "deepagents~=0.7.6" \
      "langchain~=1.3.15" \
      "langchain-openai~=1.5.1" \
      "langsmith~=0.11.2" \
      "langsmith[sandbox]~=0.11.2"

    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    with contextlib.suppress(Exception):
        os.environ["LANGSMITH_GATEWAY_API_KEY"] = userdata.get("LANGSMITH_GATEWAY_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

key = os.environ["LANGSMITH_API_KEY"]
os.environ["LANGSMITH_GATEWAY_API_KEY"] = os.environ.get("LANGSMITH_GATEWAY_API_KEY") or key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-09-sandboxes"

MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Why a sandbox and not a backend

Lesson 02 gave the agent four places to keep files. None of them can *run* anything.

That gap matters the moment an answer depends on a computation rather than on prose. Asked to
total a column, a model will produce a plausible number. Asked to run `pandas`, it produces the
right one — but only if something can execute the code it writes.

A sandbox is that something, behind a boundary. This is not a nicety: you are about to run code
that a language model wrote, and you cannot read it all first. Everything below happens on a
machine that is not yours, holds none of your credentials, and disappears when you delete it.

| | Filesystem backends (02) | Sandbox |
|---|---|---|
| Tools | `ls` `read_file` `write_file` `edit_file` `glob` `grep` | those **plus `execute`** |
| Can run code | no | yes — shell, Python, package installs |
| Lives | your process, or Postgres | a container LangSmith runs for you |
| Blast radius | your process | the box, and only the box |

---

## 2. Create the box

A **snapshot** is the reusable image; a **sandbox** is a running instance of one. Creating the
snapshot is the slow part, so we make it once and reuse it by name on every later run.

`fs_capacity_bytes` needs headroom for whatever the agent decides to `pip install`.

In [ ]:
from langsmith.sandbox import SandboxClient

SNAPSHOT = "lcw-workshop-sandbox"

client = SandboxClient()

# create_snapshot raises on a duplicate name, so reuse a ready one when it exists.
ready = [s for s in client.list_snapshots(name_contains=SNAPSHOT)
         if s.name == SNAPSHOT and s.status == "ready"]

snapshot = ready[0] if ready else client.create_snapshot(
    name=SNAPSHOT,
    docker_image="python:3.12-slim",
    fs_capacity_bytes=2 * 1024**3,
)

sb = client.create_sandbox(snapshot_id=snapshot.id)
print(f"sandbox {sb.id} — {sb.status}")

That is a whole Linux machine. Prove it before an agent goes anywhere near it:

In [ ]:
print(sb.run("cat /etc/os-release | head -2; python3 -V; whoami").stdout)

---

## 3. Hand it to an agent

`LangSmithSandbox` adapts the raw sandbox into a Deep Agents backend. Pass it as `backend=` and
the filesystem tools you already know start operating **inside the box** — plus a new one,
`execute`.

Nothing else about the agent definition changes.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import LangSmithSandbox

sandbox = LangSmithSandbox(sb)

analyst = create_deep_agent(
    model=MODEL,
    backend=sandbox,
    system_prompt=(
        "You are a data analyst working inside a sandbox. Your working directory is /work.\n"
        "Install whatever packages you need with the execute tool.\n"
        "Never estimate a number you could compute: write a script and run it.\n"
        "When asked for a report, write a single self-contained HTML file to "
        "/work/report.html with the numbers and a chart inlined as a base64 <img>."
    ),
)
analyst

### The data

Deliberately messy, and deliberately small enough to check by eye — inconsistent category
casing, stray `$` signs, two missing values. A model asked to eyeball this will get it wrong in
a way you can catch.

In [ ]:
#@title Seed the sandbox with a messy CSV (run me) { display-mode: "form" }
ORDERS_CSV = """order_date,product,category,units,revenue
2026-01-03,Standing desk,Desks,3,1287.00
2026-01-06,Desk lamp,lighting,2,$78.00
2026-01-11,Monitor arm,Accessories,2,178.00
2026-01-15,Office chair,Seating,4,996.00
2026-01-19,Keyboard tray,accessories,1,59.00
2026-02-02,Standing desk,desks,3,1287.00
2026-02-05,Desk lamp,Lighting,4,$156.00
2026-02-09,Laptop stand,Accessories,3,135.00
2026-02-12,Office chair,seating,5,1245.00
2026-02-14,Monitor arm,ACCESSORIES,6,534.00
2026-03-01,Standing desk,Desks,8,3432.00
2026-03-04,Laptop stand,accessories,5,225.00
2026-03-07,Office chair,Seating,6,$1494.00
2026-03-10,Desk lamp,lighting,5,195.00
2026-03-13,Monitor arm,Accessories,6,534.00
2026-03-19,Keyboard tray,accessories,3,177.00
2026-03-28,Standing desk,desks,,429.00
2026-03-31,Laptop stand,Accessories,4,
"""

sb.run("mkdir -p /work")
sb.write("/work/sales.csv", ORDERS_CSV)
print(sb.run("wc -l /work/sales.csv").stdout)

`sb.write` and `sb.read` move one file at a time from here. The agent does not need them — it
has the filesystem tools — but seeding an input and grabbing an artefact is your job, not its.

---

## 4. Run it

This is the first lesson in Part 1 that calls the agent from code rather than Studio, because
what we want back is a **file**, not a conversation.

In [ ]:
result = analyst.invoke({"messages": [{"role": "user", "content":
    "In /work/sales.csv are our online orders — the data is messy. Clean it, then tell me "
    "revenue by category and the monthly trend, and write me report.html."
}]})

print(result["messages"][-1].text)

Now look at what it actually did, rather than what it said it did:

In [ ]:
print(sb.run("ls -la /work").stdout)

There should be a `.py` script in there next to the report. That script is the answer to "did it
compute this or guess it" — you can read it, and re-run it.

> **Checkpoint.** Open this run's trace in LangSmith and find the first `execute` call. What did
> the agent do *before* it computed anything? A good run inspects the CSV — `head`, `dtypes`,
> a value count on `category` — because it cannot see the file from the prompt alone. An agent
> that went straight to `groupby` got lucky.

---

## 5. Serve the artefact

The agent wrote an HTML file on a machine you cannot reach. Two ways to get at it, and they are
good at different things.

**A link to one file** — `generate_download_url`. The link carries its own token, so it needs no
LangSmith credential to open. Serve HTML as `inline` and the browser renders it instead of
downloading it.

In [ ]:
link = sb.generate_download_url(
    "/work/report.html",
    content_type="text/html",
    content_disposition="inline",
    expires_in_seconds=3600,
)

print(link.download_url)

Click it. That is your agent's report, rendered.

Two things about that link that will bite you otherwise:

- **It is pinned to a path, not to contents.** Rewrite `/work/report.html` and the old link may
  serve either version. Mint a new link whenever the file changes.
- **It is a bearer credential.** Anyone holding the URL can fetch that file for the next hour.
  Do not paste it into a shared doc, and do not commit a notebook with it in the output.

---

## 6. Or serve the whole box

A single file is enough for a report. A *site* — many pages, assets, an API the page calls —
wants a real server. Start one inside the box and ask LangSmith for an authenticated URL to it.

The `wait=False` is load-bearing. `sb.run` normally blocks until the command finishes, and a web
server never finishes, so a plain call would hang the cell forever. `wait=False` hands back a
handle and leaves the server running.

In [ ]:
# wait=False: this server never exits, so do not wait for it.
server = sb.run("cd /work && python3 -m http.server 8000", wait=False, ttl_seconds=1800)

service = sb.service(port=8000, expires_in_seconds=1800)
print(service.browser_url)

That URL is a front door to a live server the agent controls, so the token in it is worth more
than the one on a file link. Both expire; neither is safe to share.

You can also drive it from Python — `service.get(...)` injects the token and refreshes it as it
ages, which is how you would smoke-test an app the agent just built:

In [ ]:
response = service.get("/report.html")
print(response.status_code, response.headers["content-type"], len(response.text), "bytes")

---

## 7. Delete it

Sandboxes idle out on their own, but a workshop should not leave a dozen boxes running.

In [ ]:
sb.delete()
print("deleted — every link above is now dead")

The snapshot survives; only the instance is gone. Re-running section 2 gives you a fresh box
from the same image in seconds.

---

## 📌 Key takeaways

- A sandbox is a **backend with a CPU** — the same filesystem tools, plus `execute`.
- The point is not convenience, it is **blast radius**: model-written code runs on a machine that holds none of your credentials and is deleted afterwards.
- **Snapshot** is the reusable image, **sandbox** the running instance. Make the snapshot once.
- `sb.run(cmd, wait=False)` for anything that does not exit. A plain `sb.run` on a web server hangs the cell until it times out.
- `generate_download_url` for one artefact, `sb.service(port=...)` for a live server. Both URLs **embed a bearer token** — they open without a LangSmith login, which is exactly why you must not share them or commit them in notebook output.
- A download link is pinned to a **path, not contents**. Rewrite the file, mint a new link.
- LangSmith sandboxes need no second credential — the key you already have is enough.

---

## ➡️ Next

**[10 · Interpreters and programmatic tool calling](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/10_interpreters_ptc.ipynb)**

The agent just ran code *against an environment*. Next: running code **inside the agent loop** —
where the interpreter becomes a wall that raw customer data cannot cross.